# Error Analysis (Thesis Sections 4.6 + 4.7)

Analyzes disagreement cases in the fine-tuned model's predictions.

**Sections:**
1. Load Fine-Tuned Results
2. Identify Disagreement Cases
3. Categorize Errors (7 Types)
4. Error Category Distribution
5. Per-Question Weakness Analysis
6. Robustness Summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score

sns.set_theme(style='whitegrid', font_scale=1.1)

RUBRICS = ['Clarity', 'Terminology', 'Coverage', 'Accuracy']

## 1. Load Fine-Tuned Results

In [ ]:
ft = pd.read_csv('../Finetuning/finetuned_results.csv')

# Normalize column names
col_map = {}
for c in ft.columns:
    if c in ('Within_1', 'Within_1.0'): col_map[c] = 'Within_1'
    elif c in ('Within_05', 'Within_0.5'): col_map[c] = 'Within_05'
ft = ft.rename(columns=col_map)

# Derived columns
ft['Diff'] = ft['LLM_Final'] - ft['Human_Final']
ft['Abs_Diff'] = np.abs(ft['Diff'])
ft['Q_num'] = ft['student_id'].str.extract(r'Q(\d+)')[0].astype(int)

print(f'Total samples: {len(ft)}')
print(f'Exact match (±0): {(ft["Abs_Diff"] == 0).sum()} ({(ft["Abs_Diff"] == 0).mean()*100:.1f}%)')
print(f'Within ±0.5: {(ft["Abs_Diff"] <= 0.5).sum()} ({(ft["Abs_Diff"] <= 0.5).mean()*100:.1f}%)')
print(f'Within ±1.0: {(ft["Abs_Diff"] <= 1.0).sum()} ({(ft["Abs_Diff"] <= 1.0).mean()*100:.1f}%)')
print(f'Outside ±1.0: {(ft["Abs_Diff"] > 1.0).sum()} ({(ft["Abs_Diff"] > 1.0).mean()*100:.1f}%)')

## 2. Identify Disagreement Cases

In [ ]:
# All cases where LLM disagrees with human (any difference)
disagree = ft[ft['Abs_Diff'] > 0].copy()
severe = ft[ft['Abs_Diff'] > 1.0].copy()

print(f'Total disagreements (|diff| > 0): {len(disagree)}')
print(f'Severe disagreements (|diff| > 1.0): {len(severe)}')
print()

# Distribution of absolute differences
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(ft['Abs_Diff'], bins=np.arange(0, ft['Abs_Diff'].max() + 0.5, 0.5),
        color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='±1.0 threshold')
ax.set_xlabel('|LLM - Human| Grade Difference', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Absolute Differences', fontsize=12)
ax.legend(fontsize=10)

ax = axes[1]
ax.hist(ft['Diff'], bins=np.arange(ft['Diff'].min() - 0.25, ft['Diff'].max() + 0.5, 0.5),
        color='#e74c3c', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linestyle='-', linewidth=1.5)
ax.set_xlabel('LLM - Human Grade Difference', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Signed Differences', fontsize=12)

plt.tight_layout()
plt.show()

## 3. Categorize Errors (7 Types)

In [ ]:
def categorize_error(row):
    """Categorize each disagreement into one of 7 error types."""
    human_ot = row['Human_OffTopic'] in ['Off-Topic', 'No Answer']
    llm_ot = row['LLM_OffTopic'] in ['Off-Topic', 'No Answer']
    diff = row['Diff']  # LLM - Human
    abs_diff = abs(diff)

    # Off-topic errors
    if not human_ot and llm_ot:
        return 'Off-Topic False Positive'
    if human_ot and not llm_ot:
        return 'Off-Topic False Negative'

    # Severe grading errors
    if diff >= 2.0:
        return 'Severe Overgrading (>=2.0)'
    if diff <= -2.0:
        return 'Severe Undergrading (>=2.0)'

    # Check rubric-specific patterns
    acc_diff = row['LLM_Accuracy'] - row['Human_Accuracy']
    cov_diff = row['LLM_Coverage'] - row['Human_Coverage']

    if abs(acc_diff) >= 2 and abs_diff > 0.5:
        return 'Accuracy Dimension Confusion'
    if cov_diff >= 1 and abs_diff > 0.5:
        return 'Coverage Over-Rewarding'

    # Partial-credit boundary
    if 0.5 < abs_diff <= 1.0:
        return 'Partial-Credit Boundary'

    return 'Minor Disagreement'


# Apply to all samples (not just disagreements)
ft['Error_Type'] = ft.apply(
    lambda row: categorize_error(row) if row['Abs_Diff'] > 0 else 'Exact Match',
    axis=1
)

print('Error categorization applied to all 510 samples.')

## 4. Error Category Distribution

In [ ]:
# Category counts (excluding exact matches for error-focused analysis)
error_counts = ft[ft['Error_Type'] != 'Exact Match']['Error_Type'].value_counts()
error_pct = (error_counts / len(ft) * 100).round(1)

error_table = pd.DataFrame({
    'Error Type': error_counts.index,
    'Count': error_counts.values,
    '% of Total': error_pct.values
})
print(f'Error Distribution (out of {len(ft)} total samples):')
print(f'Exact matches: {(ft["Error_Type"] == "Exact Match").sum()}')
print()
error_table

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
ax = axes[0]
colors_map = {
    'Off-Topic False Positive': '#e74c3c',
    'Off-Topic False Negative': '#c0392b',
    'Severe Overgrading (>=2.0)': '#e67e22',
    'Severe Undergrading (>=2.0)': '#d35400',
    'Partial-Credit Boundary': '#f39c12',
    'Accuracy Dimension Confusion': '#9b59b6',
    'Coverage Over-Rewarding': '#8e44ad',
    'Minor Disagreement': '#95a5a6'
}
bar_colors = [colors_map.get(t, '#95a5a6') for t in error_counts.index]
ax.barh(range(len(error_counts)), error_counts.values, color=bar_colors, edgecolor='white')
ax.set_yticks(range(len(error_counts)))
ax.set_yticklabels(error_counts.index, fontsize=9)
ax.set_xlabel('Count', fontsize=11)
ax.set_title('Error Category Distribution', fontsize=13, fontweight='bold')
ax.invert_yaxis()

for i, (count, pct) in enumerate(zip(error_counts.values, error_pct.values)):
    ax.text(count + 0.5, i, f'{count} ({pct}%)', va='center', fontsize=9)

# Pie chart of error types
ax = axes[1]
all_counts = ft['Error_Type'].value_counts()
pie_colors = ['#2ecc71'] + [colors_map.get(t, '#95a5a6') for t in all_counts.index if t != 'Exact Match']
ax.pie(all_counts.values, labels=all_counts.index, autopct='%1.1f%%',
       colors=pie_colors, startangle=90, textprops={'fontsize': 8})
ax.set_title('Overall Classification', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Per-Question Weakness Analysis

In [ ]:
# Per-question metrics
q_stats = []
for q in sorted(ft['Q_num'].unique()):
    mask = ft['Q_num'] == q
    q_data = ft[mask]
    n = len(q_data)
    mae = q_data['Abs_Diff'].mean()
    acc_1 = (q_data['Abs_Diff'] <= 1.0).mean() * 100
    acc_05 = (q_data['Abs_Diff'] <= 0.5).mean() * 100
    bias = q_data['Diff'].mean()
    severe_count = (q_data['Abs_Diff'] > 1.0).sum()

    h_total = (q_data['Human_Clarity'] + q_data['Human_Terminology'] +
               q_data['Human_Coverage'] + q_data['Human_Accuracy']).astype(int)
    l_total = (q_data['LLM_Clarity'] + q_data['LLM_Terminology'] +
               q_data['LLM_Coverage'] + q_data['LLM_Accuracy']).astype(int).clip(0, 10)
    try:
        qwk = cohen_kappa_score(h_total, l_total, weights='quadratic')
    except:
        qwk = np.nan

    q_stats.append({
        'Question': f'Q{q}', 'N': n, 'MAE': round(mae, 3),
        'Acc ±1.0 (%)': round(acc_1, 1), 'Acc ±0.5 (%)': round(acc_05, 1),
        'QWK': round(qwk, 3), 'Bias': round(bias, 3),
        'Severe Errors': severe_count
    })

q_df = pd.DataFrame(q_stats).sort_values('MAE', ascending=False)
print('Per-Question Performance (sorted by MAE, worst first):')
q_df

In [ ]:
# Highlight weakest questions
weak_qs = q_df.head(3)
print('Top 3 Weakest Questions:')
print('='*50)
for _, row in weak_qs.iterrows():
    print(f"\n{row['Question']} (N={row['N']})")
    print(f"  MAE: {row['MAE']}, QWK: {row['QWK']}")
    print(f"  ±1.0 Acc: {row['Acc ±1.0 (%)']}%, ±0.5 Acc: {row['Acc ±0.5 (%)']}%")
    print(f"  Bias: {row['Bias']:+.3f}, Severe errors: {row['Severe Errors']}")

In [ ]:
# Detailed error breakdown for weak questions
weak_q_nums = [int(q.replace('Q', '')) for q in weak_qs['Question'].values]

for q in weak_q_nums:
    mask = ft['Q_num'] == q
    q_errors = ft[mask & (ft['Abs_Diff'] > 0)]
    print(f'\nQ{q} Error Type Breakdown ({len(q_errors)} disagreements out of {mask.sum()}):')
    error_dist = q_errors['Error_Type'].value_counts()
    for error_type, count in error_dist.items():
        print(f'  {error_type:35s}: {count}')

In [ ]:
# Per-question performance visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

q_sorted = q_df.sort_values('Question')
questions = q_sorted['Question'].values
x = np.arange(len(questions))

# MAE by question
ax = axes[0]
colors = ['#e74c3c' if mae > q_sorted['MAE'].median() else '#3498db' for mae in q_sorted['MAE']]
ax.bar(x, q_sorted['MAE'], color=colors, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(questions, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('MAE', fontsize=11)
ax.set_title('MAE by Question', fontsize=13, fontweight='bold')
ax.axhline(q_sorted['MAE'].median(), color='gray', linestyle='--', alpha=0.7, label='Median')
ax.legend()

# QWK by question
ax = axes[1]
colors = ['#2ecc71' if qwk > q_sorted['QWK'].median() else '#e74c3c' for qwk in q_sorted['QWK']]
ax.bar(x, q_sorted['QWK'], color=colors, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(questions, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('QWK', fontsize=11)
ax.set_title('QWK by Question', fontsize=13, fontweight='bold')
ax.axhline(q_sorted['QWK'].median(), color='gray', linestyle='--', alpha=0.7, label='Median')
ax.legend()

plt.tight_layout()
plt.show()

## 6. Robustness Summary

In [ ]:
print('='*60)
print('ROBUSTNESS SUMMARY')
print('='*60)

total = len(ft)
exact = (ft['Abs_Diff'] == 0).sum()
within_05 = (ft['Abs_Diff'] <= 0.5).sum()
within_1 = (ft['Abs_Diff'] <= 1.0).sum()
severe = (ft['Abs_Diff'] > 1.0).sum()
very_severe = (ft['Abs_Diff'] >= 2.0).sum()

print(f'\nOverall Accuracy Profile ({total} samples):')
print(f'  Exact match:      {exact:4d}  ({exact/total*100:.1f}%)')
print(f'  Within ±0.5:      {within_05:4d}  ({within_05/total*100:.1f}%)')
print(f'  Within ±1.0:      {within_1:4d}  ({within_1/total*100:.1f}%)')
print(f'  Outside ±1.0:     {severe:4d}  ({severe/total*100:.1f}%)')
print(f'  Severe (>=2.0):   {very_severe:4d}  ({very_severe/total*100:.1f}%)')

# Off-topic summary
human_ot = ft['Human_OffTopic'].isin(['Off-Topic', 'No Answer'])
llm_ot = ft['LLM_OffTopic'].isin(['Off-Topic', 'No Answer'])
ot_fp = (~human_ot & llm_ot).sum()
ot_fn = (human_ot & ~llm_ot).sum()

print(f'\nOff-Topic Detection:')
print(f'  Total off-topic (human): {human_ot.sum()}')
print(f'  False Positives: {ot_fp}')
print(f'  False Negatives: {ot_fn}')

# Bias analysis
mean_bias = ft['Diff'].mean()
print(f'\nGrading Bias:')
print(f'  Mean bias (LLM - Human): {mean_bias:+.4f}')
print(f'  Direction: {"Slight overgrading" if mean_bias > 0 else "Slight undergrading"}')
print(f'  Median difference: {ft["Diff"].median():+.1f}')

# Cross-question consistency
q_accs = [float(row['Acc ±1.0 (%)']) for _, row in q_df.iterrows()]
print(f'\nCross-Question Consistency:')
print(f'  ±1.0 Acc range: {min(q_accs):.1f}% — {max(q_accs):.1f}%')
print(f'  ±1.0 Acc std:   {np.std(q_accs):.1f}%')
print(f'  All questions above 60% ±1.0 acc: {"Yes" if min(q_accs) > 60 else "No"}')

In [ ]:
# Per-rubric error analysis
print('\nPer-Rubric Error Profile:')
for rubric in RUBRICS:
    h = ft[f'Human_{rubric}'].values
    l = ft[f'LLM_{rubric}'].values
    diff = l - h
    abs_diff = np.abs(diff)
    max_score = 4 if rubric == 'Accuracy' else 2
    print(f'\n  {rubric} (0-{max_score}):')
    print(f'    Exact match: {(abs_diff == 0).mean()*100:.1f}%')
    print(f'    Mean bias:   {diff.mean():+.3f}')
    print(f'    MAE:         {abs_diff.mean():.3f}')